In [2]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

# Define a simplified QFNN circuit (2 qubits, 1 layer for illustration)
def qfnn_circuit(x, w_enc, w_int):
    # Phase encoding
    for i in range(2):
        qml.RY(w_enc[i, 0] * x[i], wires=i)
        qml.RZ(w_enc[i, 1] * x[i], wires=i)
    # Interference layer
    qml.CNOT(wires=[0, 1])
    qml.CRX(w_int[0, 0], wires=[0, 1])
    qml.CRY(w_int[0, 1], wires=[0, 1])
    return qml.expval(qml.PauliZ(0))

# Define a simplified QBPNN circuit (with residual connection)
def qbpnn_circuit(x, w_enc, w_int):
    # Phase encoding
    for i in range(2):
        qml.RY(w_enc[i, 0] * x[i], wires=i)
        qml.RZ(w_enc[i, 1] * x[i], wires=i)
    # Interference layer
    qml.CNOT(wires=[0, 1])
    qml.CRX(w_int[0, 0], wires=[0, 1])
    qml.CRY(w_int[0, 1], wires=[0, 1])
    # Residual connection
    for i in range(2):
        qml.RY(x[i], wires=i)
    return qml.expval(qml.PauliZ(0))

# Initialize parameters
n_qubits = 2
dev = qml.device("lightning.qubit", wires=n_qubits)
x = np.array([0.5, 0.5])  # Dummy input
w_enc = np.random.uniform(-np.pi, np.pi, (2, 2))
w_int = np.random.uniform(-np.pi, np.pi, (1, 2))

# Create QFNN circuit
qfnn = qml.QNode(qfnn_circuit, dev)
fig1, ax1 = qml.draw_mpl(qfnn)(x, w_enc, w_int)
fig1.suptitle("QFNN Circuit", fontsize=12)
plt.savefig("../figures/qfnn_circuit.png", dpi=800, bbox_inches="tight")
plt.close()

# Create QBPNN circuit
qbpnn = qml.QNode(qbpnn_circuit, dev)
fig2, ax2 = qml.draw_mpl(qbpnn)(x, w_enc, w_int)
fig2.suptitle("QBPNN Circuit", fontsize=12)
plt.savefig("../figures/qbpnn_circuit.png", dpi=800, bbox_inches="tight")
plt.close()

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

# Data from Table 1 (All configuration)
data = {
    "Dataset": ["Moons", "Circles", "Linear Blobs", "Gaussian Quantiles", "Iris (2D)", "XOR"],
    "QFNN_Accuracy": [0.4333, 0.7000, 0.5000, 0.4000, 0.6667, 0.5333],
    "QBPNN_Accuracy": [0.8000, 0.9667, 1.0000, 0.8333, 0.8333, 1.0000]
}
df = pd.DataFrame(data)

# Plotting
fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.35
x = range(len(df["Dataset"]))
ax.bar([i - bar_width/2 for i in x], df["QFNN_Accuracy"], bar_width, label="QFNN", color="blue")
ax.bar([i + bar_width/2 for i in x], df["QBPNN_Accuracy"], bar_width, label="QBPNN", color="orange")
ax.set_xlabel("Dataset")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy of QFNN and QBPNN (All Configuration)")
ax.set_xticks(x)
ax.set_xticklabels(df["Dataset"], rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig("../figures/performance_bar.png", dpi=800, bbox_inches="tight")
plt.close()
